# 08 — Self-RAG: Retrieve, Generate, and Critique

## Goal
This workshop notebook demonstrates a simple Self-RAG-inspired loop. The system retrieves evidence, drafts an answer, then asks the model to critique whether the answer is supported by the evidence and whether more retrieval is needed.

This is a **teaching approximation**, not a faithful implementation of the full Self-RAG research architecture. Model-generated critiques can be wrong, so treat them as diagnostics—not guarantees.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first.

## Architecture

```mermaid
flowchart TD
    A["Question"] --> B["Retrieve chunks"]
    B --> C["Draft grounded answer"]
    C --> D["Self-critique"]
    D --> E{"Evidence sufficient?"}
    E -->|"No"| F["Retrieve more / broaden query"]
    F --> C
    E -->|"Yes"| G["Return answer + sources"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef retrieval fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef critique fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef decision fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef output fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A,G input;
    class B,F retrieval;
    class C output;
    class D critique;
    class E decision;
```

## 1. Imports and paths

In [ ]:
from pathlib import Path
import re
from collections import OrderedDict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vector_store").exists() and (PROJECT_ROOT.parent / "vector_store").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Open the existing Chroma store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Connected to:", COLLECTION_NAME)
print("Stored record count:", vector_store._collection.count())

## 4. Retrieval and source helpers

In [ ]:
def doc_identity(doc):
    meta = doc.metadata or {}
    return (
        str(meta.get("source", "unknown")),
        meta.get("page"),
        meta.get("start_index"),
        doc.page_content[:180],
    )

def deduplicate_documents(documents):
    unique = OrderedDict()
    for doc in documents:
        unique.setdefault(doc_identity(doc), doc)
    return list(unique.values())

def source_label(doc, index: int) -> str:
    meta = doc.metadata or {}
    source = Path(str(meta.get("source", "unknown source"))).name
    page = meta.get("page", "unknown")
    start = meta.get("start_index")
    label = f"[S{index}] {source}, page {page}"
    if start is not None:
        label += f", chunk start {start}"
    return label

def build_context(documents, max_chars_per_doc: int = 1800) -> str:
    return "\\n\\n---\\n\\n".join(
        f"{source_label(doc, i)}\\n{doc.page_content.strip()[:max_chars_per_doc]}"
        for i, doc in enumerate(documents, start=1)
    )

def retrieve(question: str, k: int = 5):
    return vector_store.similarity_search(question, k=k)


## 5. Draft and critique prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

draft_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's Windows security benchmark question using only the context. "
     "Cite claims using source labels such as [S1]. State when evidence is missing. "
     "Do not claim that benchmark text proves a real machine is compliant."),
    ("human", "Question: {question}\\n\\nContext:\\n{context}")
])

critique_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Critique the draft answer against the supplied context. Return exactly these fields: "
     "SUPPORTED: YES or NO; MISSING: short explanation; RETRIEVE_MORE: YES or NO. "
     "Use NO for SUPPORTED if important claims lack support. Do not rewrite the answer."),
    ("human", "Question: {question}\\n\\nContext:\\n{context}\\n\\nDraft answer:\\n{answer}")
])

def ask_model(prompt, **kwargs):
    response = (prompt | llm).invoke(kwargs)
    return response.content if hasattr(response, "content") else str(response)

def self_rag(question: str, k: int = 5, max_rounds: int = 2) -> dict:
    documents = retrieve(question, k=k)
    rounds = []
    for round_no in range(1, max_rounds + 1):
        context = build_context(documents)
        answer = ask_model(draft_prompt, question=question, context=context or "[No context retrieved.]")
        critique = ask_model(
            critique_prompt, question=question, context=context or "[No context retrieved.]",
            answer=answer
        )
        rounds.append({
            "round": round_no, "document_count": len(documents),
            "answer": answer, "critique": critique
        })
        # The model's critique is heuristic; stop if it appears satisfied.
        if "RETRIEVE_MORE: YES" not in critique.upper() or round_no == max_rounds:
            break
        broader_query = (
            "CIS Microsoft Windows 11 Enterprise benchmark details and assessment guidance for: "
            + question
        )
        documents = deduplicate_documents(
            documents + retrieve(broader_query, k=k)
        )[:max(k, 8)]
    return {"question": question, "documents": documents, "rounds": rounds,
            "answer": rounds[-1]["answer"], "critique": rounds[-1]["critique"]}

## 6. Run Self-RAG

In [ ]:
question = "What does the benchmark recommend about screen locking?"
result = self_rag(question, k=5, max_rounds=2)

print("FINAL ANSWER\\n")
print(result["answer"])
print("\\nSELF-CRITIQUE\\n")
print(result["critique"])
print("\\nROUNDS")
for item in result["rounds"]:
    print(f"Round {item['round']}: {item['document_count']} chunks")

## 7. Inspect final evidence

In [ ]:
for i, doc in enumerate(result["documents"], start=1):
    print("=" * 88)
    print(source_label(doc, i))
    print(doc.page_content[:900])
    print()

## 8. Experiment

Try a question that may not be covered by the benchmark. Check whether the critique catches unsupported claims. It may not; verify the source chunks yourself.

In [ ]:
experiment_question = "What is the organization's current Windows 11 compliance status?"
experiment = self_rag(experiment_question, k=5, max_rounds=2)
print(experiment["answer"])
print("\\nCritique:")
print(experiment["critique"])

## Summary and limitations

- Self-critique can reveal unsupported claims or missing evidence.
- A second retrieval round can add context when the critique requests it.
- The critique is generated by the same model and may be self-confirming or incorrect.
- This notebook does not implement trained reflection tokens or the full research method.
- Validate claims against retrieved text and use trusted endpoint evidence for actual compliance assessment.